In [1]:
import torch
from transformers import BertTokenizer, BertModel

In [2]:
# 1. 加载预训练模型和分词器
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# 2. 定义一个测试句子
test_sentence = "This piece of music sounds relaxing."

# 3. 使用分词器对测试句子进行编码，返回 PyTorch tensors
inputs = tokenizer(test_sentence, return_tensors="pt")

# 4. 通过模型进行前向计算，获取输出
# 注意：这里我们用 no_grad()，避免计算梯度，节省内存
with torch.no_grad():
    outputs = model(**inputs)

# 5. 获取最后一层的隐藏状态（batch_size, sequence_length, hidden_size）
last_hidden_state = outputs.last_hidden_state

print("隐藏状态的形状:", last_hidden_state.shape)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


隐藏状态的形状: torch.Size([1, 9, 768])


In [87]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
# 定义模板句和目标词
template_sentence = "This piece of music sounds [MASK]."
target_word = "warm and a little bit wet"

# 用目标词替换模板句中的 [MASK]
filled_sentence = template_sentence.replace("[MASK]", target_word)
print("替换后的句子:", filled_sentence)

# 使用分词器进行编码，同时返回token的偏移信息（有助于定位目标词）
inputs = tokenizer(filled_sentence, return_tensors="pt", return_offsets_mapping=True)
offset_mapping = inputs.pop("offset_mapping")  # 从输入中取出offset信息

# 获取token列表
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
print("分词结果:", tokens)

# 分析目标词在分词列表中的位置
# 一种简单方法是查看哪些token的文本部分包含目标词的一部分
target_token_ids = tokenizer(target_word, add_special_tokens=False)["input_ids"]
target_tokens = tokenizer.convert_ids_to_tokens(target_token_ids)
print("目标词可能对应的token:", target_tokens)

# 方法1：通过查找连续子序列匹配（注意：由于大小写和子词标记“##”的存在，匹配需要处理这些情况）
def find_sublist(tokens, sub_tokens):
    n = len(sub_tokens)
    for i in range(len(tokens) - n + 1):
        # 去除前缀"##"后再比对
        if [t.lstrip("##") for t in tokens[i:i+n]] == [st.lstrip("##") for st in sub_tokens]:
            return i, i+n
    return None

match = find_sublist(tokens, target_tokens)
if match:
    start_idx, end_idx = match
    print(f"目标词在token列表中的索引区间: {start_idx} 到 {end_idx-1}")
else:
    print("未在token列表中找到目标词的连续token序列。")



替换后的句子: This piece of music sounds warm and a little bit wet.
分词结果: ['[CLS]', 'this', 'piece', 'of', 'music', 'sounds', 'warm', 'and', 'a', 'little', 'bit', 'wet', '.', '[SEP]']
目标词可能对应的token: ['warm', 'and', 'a', 'little', 'bit', 'wet']
目标词在token列表中的索引区间: 6 到 11


In [88]:
# 用模型计算得到隐藏状态
with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state  # 形状: (1, sequence_length, hidden_size)
print("隐藏状态的形状:", last_hidden_state.shape)

# 提取目标词对应的token向量，并求平均
if match:
    # 注意这里 squeeze 后维度为 (sequence_length, hidden_size)
    target_vectors = last_hidden_state.squeeze()[start_idx:end_idx]
    print(target_vectors.shape)
    avg_vector = target_vectors.mean(dim=0)
    print("目标词/短语的平均向量形状:", avg_vector.shape)

隐藏状态的形状: torch.Size([1, 14, 768])
torch.Size([6, 768])
目标词/短语的平均向量形状: torch.Size([768])


In [89]:
avg_vector

tensor([-4.5907e-01, -3.7284e-01,  2.6824e-01,  4.0523e-01, -2.1251e-01,
         2.2506e-01, -1.6994e-01, -1.0682e-01,  6.3423e-02, -1.9893e-01,
         6.2298e-01, -3.0936e-01, -6.1058e-02,  8.7082e-01, -2.3974e-01,
         5.7228e-01,  1.9898e-01, -1.9546e-01,  3.1596e-01,  4.6010e-01,
         3.2079e-01, -2.7677e-01, -8.5889e-01,  8.5149e-01,  4.4705e-01,
        -1.1331e-01, -1.0588e-01, -8.7015e-02,  6.7179e-03, -3.3243e-02,
         7.4687e-01,  9.1954e-02,  2.4284e-02, -1.8500e-01, -4.0211e-01,
        -2.0087e-01, -7.3337e-02, -2.6264e-01, -5.2921e-01,  8.7437e-02,
        -7.1026e-01, -3.6878e-01, -4.3237e-01, -1.6795e-01, -1.7243e-02,
        -2.1803e-01,  4.6858e-01, -2.7176e-01, -4.1526e-01, -5.1694e-01,
         2.2680e-01, -4.1506e-01, -2.7317e-01, -2.5298e-01,  2.8225e-01,
         5.0876e-01,  3.7347e-01, -6.5608e-01, -4.7929e-01,  8.3925e-02,
        -5.8221e-02,  1.0977e-01,  8.7295e-02, -1.1505e+00,  1.0869e-01,
         2.7554e-01, -1.1075e-01, -2.9218e-01, -5.1

In [91]:

import torch.nn.functional as F



def extract_target_vector(template, target_word):
    """
    给定模板句和目标词，替换占位符，计算目标词的平均向量表示
    """
    # 替换模板中的 [MASK] 占位符
    filled_sentence = template.replace("[MASK]", target_word)
    # 对句子进行编码，并返回offset mapping（用于定位目标词）
    inputs = tokenizer(filled_sentence, return_tensors="pt", return_offsets_mapping=True)
    offset_mapping = inputs.pop("offset_mapping")
    
    # 获取token列表
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
    # 对目标词单独编码，得到它对应的token序列
    target_token_ids = tokenizer(target_word, add_special_tokens=False)["input_ids"]
    target_tokens = tokenizer.convert_ids_to_tokens(target_token_ids)
    
    # 在完整句子中查找目标词的连续token位置
    def find_sublist(tokens, sub_tokens):
        n = len(sub_tokens)
        for i in range(len(tokens) - n + 1):
            if [t.lstrip("##") for t in tokens[i:i+n]] == [st.lstrip("##") for st in sub_tokens]:
                return i, i+n
        return None
    
    match = find_sublist(tokens, target_tokens)
    if not match:
        raise ValueError("在句子中没有找到目标词对应的token序列。")
    start_idx, end_idx = match

    # 前向传递获得隐藏状态
    with torch.no_grad():
        outputs = model(**inputs)
    last_hidden_state = outputs.last_hidden_state  # (1, sequence_length, hidden_size)

    # 去掉batch维度，提取目标词token对应的向量，并求平均
    target_vectors = last_hidden_state.squeeze()[start_idx:end_idx]
    avg_vector = target_vectors.mean(dim=0)
    return avg_vector

# 定义模板句，这里我们使用同样的模板
template_sentence = "I make this music sound [MASK] by manipulating its frequency content."

word1 = "warm"
word2 = "cold"
vector1 = extract_target_vector(template_sentence, word1)
# lively 的向量，lively 作为 energetic 的近义词
vector2 = extract_target_vector(template_sentence, word2)

# 计算余弦相似度（cosine similarity）
cos_sim = F.cosine_similarity(vector1.unsqueeze(0), vector2.unsqueeze(0))
print(word1, "和", word2, "向量的余弦相似度：", cos_sim.item())

warm 和 cold 向量的余弦相似度： 0.7551626563072205


In [86]:
# 定义多个模板句（根据需求可以设计更多）
templates = [
    "This piece of music exhibits a [MASK] timbre that shapes its overall emotional character.",
    "The sound texture of this track is distinctly [MASK], giving it a unique mood.",
    "The overall vibe of this music feels [MASK].",
    "The sound of this track is [MASK]",
    "I want to make the timbre of this audio sounds [MASK].",
    "The producer has done lots of work and processed this audio to make it [MASK]!"
]


# 对每个模板计算目标词向量
def TakeAvgVec(target_word, templates):
    vectors = []
    for template in templates:
        vec = extract_target_vector(template, target_word)
        vectors.append(vec)

    # 将所有模板得到的向量求平均
    final_vector = torch.stack(vectors, dim=0).mean(dim=0)
    return final_vector

target_word1 = "warm"
target_word2 = "cold"
vector1 = TakeAvgVec(target_word1,templates)
vector2 = TakeAvgVec(target_word2,templates)
cos_sim = F.cosine_similarity(vector1.unsqueeze(0), vector2.unsqueeze(0))
print(target_word1, "和", target_word2, "向量的余弦相似度：", cos_sim.item())

warm 和 cold 向量的余弦相似度： 0.7418567538261414
